In [6]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [8]:
from src.data_utils import load_raw


## 1. application_train Overview

In [9]:
df=load_raw("application_train.csv")

In [22]:
df.shape

(307511, 122)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), object(16)
memory usage: 286.2+ MB


In [34]:
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


- The application_train dataset contains 307511 rows and 122 columns.
- The target variable ```TARGET``` represents binary outcome, where 1 indicates default and 0 indicates non-default.

In [33]:
df['TARGET'].value_counts(normalize=True)

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Approximately 8% of observation correspond to default cases, confirming strong imbalance.

## 2. Key Column Identification
1. Primary Key: SK_ID_CURR
2. Target Column: TARGET

SK_ID_CURR uniquely identifies each loan application and it will serve as the central join key across relational tables.

## 3. Bureau Relationship Analysis

In [27]:
bureau=load_raw("bureau.csv")

bureau.shape

(1716428, 17)

In [28]:
bureau['SK_ID_CURR'].nunique()

305811

- application_train customers: 307,511
- bureau unique customers: 305,811

Since the total rows (1.7M) greatly exceed unique customers (305k), this confirms a 1-to-many relationship between customer and bureau records.

## 4. Data Leakage Awareness
- When aggregating historical tables such as bureau or installments, we must ensure that only information available at the time of loan application is used.

- Including future payment behavior would introduce data leakage and artificially inflate model performance.

## 5. Relationship Summary

- application_train → Base table
- bureau → 1-to-many
- previous_application → 1-to-many
- installments_payments → 1-to-many
- credit_card_balance → 1-to-many

All secondary tables must be aggregated at the SK_ID_CURR level
before merging to prevent row duplication.

## 6. Aggregation Strategy Draft

```bash
bureau_agg=bureau.groupby('SK_ID_CURR').agg({
    'DAYS_CREDIT': ['min','max','mean'],
    'AMT_CREDIT_SUM': ['sum','mean']
})
```